## Multi-Tool System with the Claude API

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Create the Anthropic Client

In [ ]:
import anthropic 

client = anthropic.Anthropic(api_key = claude_api_key)

### Implement the Weather API Call

In [ ]:
import requests

def get_current_weather(location: str):

    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return response.json()

### Implement the Agent Loop

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Give me code snippet to create Foundry Client using SDK via MSLearn Docs?"
    }
]

while True:
    response = client.beta.messages.create(
        model = claude_model_name,
        max_tokens = 4096,
        betas = ["mcp-client-2025-11-20"],
        messages = messages,
        mcp_servers = [
            {
                        "type": "url",
                        "url": "https://learn.microsoft.com/api/mcp",
                        "name": "MSLearnMCPServer"
            }
        ],
        tools = [
            {"type": "mcp_toolset", "mcp_server_name": "MSLearnMCPServer"},
            {
                        "name": "GetCurrentWeather",
                        "description": "Retrieve the current weather information for a specified location using the wttr.in weather service.",
                        "input_schema": {
                            "type": "object",
                            "properties": {
                                "location": {
                                    "type": "string",
                                    "description": "City or location to retrieve the weather for, for example London, New York, or Mumbai."
                                },
                            },
                            "required": ["location"]
                    }
            },
        ]
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content
        }
    )

    tool_used = False

    for block in response.content:
        if block.type == "text":
            print(block.text + "\n\n")

        elif block.type == "tool_use":

            tool_used = True

            if block.name == "GetCurrentWeather":

                print("Invoking GetCurrentWeather Tool \n\n")
                weather = get_current_weather(
                    block.input["location"]
                )

                messages.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": str(weather)
                            }
                        ]
                    }
                )

    if not tool_used:
        break